# STEP 5 — AI Strategic Insight Engine

Tahap ini mengubah hasil analisis dan Machine Learning menjadi executive insights dan rekomendasi strategis otomatis untuk mendukung pengambilan keputusan berbasis data.

Output utama:
1. Program Performance KPI
2. Budget Efficiency Analytics
3. Government Program Status
4. Strategic Recommendations
5. Executive Summary (JSON)

In [3]:
# =========================================================
# STEP 5 — LOAD DATA AND MODELS (ROBUST VERSION)
# =========================================================

import pandas as pd
import numpy as np
import json
import joblib
from pathlib import Path

# =========================================================
# 1. LOAD CLEANED DATASET
# =========================================================

dataset_path = Path("../data/processed/program_impact_cleaned.csv")

if not dataset_path.exists():
    raise FileNotFoundError(
        f"Dataset not found: {dataset_path}\n"
        "Please complete STEP 2 first."
    )

df = pd.read_csv(
    dataset_path,
    parse_dates=["start_date"]
)

print("Dataset loaded successfully.")
print("Dataset Shape:", df.shape)

# =========================================================
# 2. LOAD MODELS (OPTIONAL)
# =========================================================

impact_model_path = Path("../models/impact_classifier.pkl")
effectiveness_model_path = Path("../models/effectiveness_regressor.pkl")

impact_model = None
effectiveness_model = None

# Impact Classification Model
if impact_model_path.exists():
    impact_model = joblib.load(impact_model_path)
    print("Impact classification model loaded successfully.")
else:
    print("⚠️ impact_classifier.pkl not found.")
    print("   STEP 4 should be completed before loading the model.")

# Effectiveness Regression Model
if effectiveness_model_path.exists():
    effectiveness_model = joblib.load(effectiveness_model_path)
    print("Effectiveness regression model loaded successfully.")
else:
    print("⚠️ effectiveness_regressor.pkl not found.")
    print("   STEP 4 should be completed before loading the model.")

# =========================================================
# 3. LOAD STEP 3 SUMMARY (OPTIONAL)
# =========================================================

summary_path = Path("../data/processed/program_impact_summary.json")

if summary_path.exists():
    with open(summary_path, "r", encoding="utf-8") as f:
        eda_summary = json.load(f)
    print("EDA summary loaded successfully.")
else:
    eda_summary = {}
    print("⚠️ program_impact_summary.json not found.")

# =========================================================
# 4. FINAL STATUS
# =========================================================

print("\n" + "=" * 60)
print("STEP 5 INITIALIZATION COMPLETED")
print("=" * 60)
print(f"Dataset Loaded      : {df.shape}")
print(f"Impact Model        : {'Available' if impact_model is not None else 'Not Available'}")
print(f"Regression Model    : {'Available' if effectiveness_model is not None else 'Not Available'}")
print(f"EDA Summary         : {'Available' if eda_summary else 'Not Available'}")
print("=" * 60)

Dataset loaded successfully.
Dataset Shape: (10000, 26)
Impact classification model loaded successfully.
Effectiveness regression model loaded successfully.
EDA summary loaded successfully.

STEP 5 INITIALIZATION COMPLETED
Dataset Loaded      : (10000, 26)
Impact Model        : Available
Regression Model    : Available
EDA Summary         : Available


In [4]:
# =========================================================
# 5. EXECUTIVE KPI CALCULATION
# =========================================================

total_programs = len(df)
total_budget = df["budget_allocated"].sum()
avg_effectiveness = df["effectiveness_score"].mean()
avg_roi = df["roi_score"].mean()
avg_satisfaction = df["satisfaction_score"].mean()
avg_budget_utilization = df["budget_utilization"].mean()
high_impact_rate = (
    (df["impact_category"] == "High Impact")
    .mean() * 100
)

top_program = (
    df.groupby("program_name")["effectiveness_score"]
    .mean()
    .sort_values(ascending=False)
    .index[0]
)

top_department = (
    df.groupby("department")["effectiveness_score"]
    .mean()
    .sort_values(ascending=False)
    .index[0]
)

top_district = (
    df.groupby("district")["effectiveness_score"]
    .mean()
    .sort_values(ascending=False)
    .index[0]
)

# =========================================================
# 6. GOVERNMENT PROGRAM STATUS
# =========================================================

if avg_effectiveness >= 85:
    program_status = "Excellent Performance"
    status_color = "#10B981"
elif avg_effectiveness >= 75:
    program_status = "Strong Performance"
    status_color = "#3B82F6"
elif avg_effectiveness >= 65:
    program_status = "Moderate Performance"
    status_color = "#F59E0B"
else:
    program_status = "Strategic Intervention Required"
    status_color = "#EF4444"

# =========================================================
# 7. STRATEGIC RECOMMENDATIONS
# =========================================================

recommendations = []

if avg_effectiveness < 80:
    recommendations.append(
        "Improve program design and implementation effectiveness."
    )

if avg_budget_utilization < 85:
    recommendations.append(
        "Optimize budget execution to increase resource efficiency."
    )

if avg_satisfaction < 4.0:
    recommendations.append(
        "Strengthen citizen engagement to improve satisfaction."
    )

if high_impact_rate < 50:
    recommendations.append(
        "Scale high-performing programs and redesign low-impact initiatives."
    )

if avg_roi < 80:
    recommendations.append(
        "Reallocate budget toward programs with higher return potential."
    )

if len(recommendations) == 0:
    recommendations = [
        "Maintain current strategy and expand top-performing programs.",
        "Document best practices for replication across departments.",
        "Continue monitoring performance indicators."
    ]

# =========================================================
# 8. DEPARTMENT PERFORMANCE RANKING
# =========================================================

department_performance = (
    df.groupby("department")
    .agg({
        "effectiveness_score": "mean",
        "roi_score": "mean",
        "satisfaction_score": "mean",
        "budget_allocated": "sum"
    })
    .round(2)
    .sort_values("effectiveness_score", ascending=False)
)

# =========================================================
# 9. TOP 10 PROGRAMS
# =========================================================

top_10_programs = (
    df.sort_values("effectiveness_score", ascending=False)
    [
        [
            "program_name",
            "department",
            "district",
            "budget_allocated",
            "beneficiaries",
            "effectiveness_score",
            "roi_score",
            "impact_category"
        ]
    ]
    .head(10)
)

# =========================================================
# 10. EXECUTIVE SUMMARY JSON
# =========================================================

executive_summary = {
    "total_programs": int(total_programs),
    "total_budget_allocated": float(total_budget),
    "average_effectiveness_score": round(avg_effectiveness, 2),
    "average_roi_score": round(avg_roi, 2),
    "average_satisfaction_score": round(avg_satisfaction, 2),
    "average_budget_utilization": round(avg_budget_utilization, 2),
    "high_impact_program_rate": round(high_impact_rate, 2),
    "top_program": top_program,
    "top_department": top_department,
    "top_district": top_district,
    "government_program_status": program_status,
    "strategic_recommendations": recommendations
}

# =========================================================
# 11. SAVE EXECUTIVE SUMMARY
# =========================================================

Path("../data/processed").mkdir(parents=True, exist_ok=True)

with open(
    "../data/processed/final_program_impact_summary.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        executive_summary,
        f,
        indent=4,
        ensure_ascii=False
    )

print("Executive summary saved successfully.")
print("../data/processed/final_program_impact_summary.json")

# =========================================================
# 12. EXECUTIVE DASHBOARD SUMMARY
# =========================================================

print("=" * 80)
print("GOVERNMENT PROGRAM IMPACT EXECUTIVE SUMMARY")
print("=" * 80)
print(f"Total Programs              : {total_programs:,}")
print(f"Total Budget Allocated      : Rp {total_budget:,.0f}")
print(f"Average Effectiveness Score : {avg_effectiveness:.2f}")
print(f"Average ROI Score           : {avg_roi:.2f}")
print(f"Average Satisfaction Score  : {avg_satisfaction:.2f} / 5.00")
print(f"High Impact Program Rate    : {high_impact_rate:.2f}%")
print()
print(f"Top Program                 : {top_program}")
print(f"Top Department              : {top_department}")
print(f"Top District                : {top_district}")
print(f"Government Program Status   : {program_status}")
print("=" * 80)

print("\nSTRATEGIC RECOMMENDATIONS")
print("-" * 80)

for i, rec in enumerate(recommendations, 1):
    print(f"{i}. {rec}")

print("=" * 80)

# =========================================================
# 13. DISPLAY TABLES
# =========================================================

print("\n🏛️ Department Performance Ranking")
display(department_performance)

print("\n🏆 Top 10 High-Impact Programs")
display(top_10_programs)

# =========================================================
# 14. DISPLAY JSON OUTPUT
# =========================================================

print("\n📄 Executive Summary (JSON Output)")
print(json.dumps(
    executive_summary,
    indent=4,
    ensure_ascii=False
))

Executive summary saved successfully.
../data/processed/final_program_impact_summary.json
GOVERNMENT PROGRAM IMPACT EXECUTIVE SUMMARY
Total Programs              : 10,000
Total Budget Allocated      : Rp 51,788,261,574,851
Average Effectiveness Score : 80.92
Average ROI Score           : 80.92
Average Satisfaction Score  : 4.09 / 5.00
High Impact Program Rate    : 22.80%

Top Program                 : Environmental Sustainability
Top Department              : Dinas Lingkungan Hidup
Top District                : Bumiaji
Government Program Status   : Strong Performance

STRATEGIC RECOMMENDATIONS
--------------------------------------------------------------------------------
1. Scale high-performing programs and redesign low-impact initiatives.

🏛️ Department Performance Ranking


,effectiveness_score,roi_score,satisfaction_score,budget_allocated
department,,,,
Dinas Lingkungan Hidup,81.21,81.21,4.07,5096915993832
Dinas Pendidikan,81.20,81.20,4.11,5331105426200
Dinas Pariwisata,81.15,81.15,4.12,5118166063972
Dinas Koperasi,80.92,80.92,4.11,10205744759681
Dinas Pertanian,80.87,80.87,4.07,4930920977599
Dinas Kesehatan,80.80,80.80,4.07,5424720591488
DISKOMINFO,80.71,80.71,4.09,15680687762079



🏆 Top 10 High-Impact Programs


,program_name,department,district,budget_allocated,beneficiaries,effectiveness_score,roi_score,impact_category
9172,Healthcare Outreach,Dinas Kesehatan,Bumiaji,7582050204,18569,98.17,98.17,High Impact
3884,Healthcare Outreach,Dinas Kesehatan,Junrejo,9235058532,28590,96.84,96.84,High Impact
1911,Education Assistance,Dinas Pendidikan,Batu,4238550295,38567,96.41,96.41,High Impact
2156,Education Assistance,Dinas Pendidikan,Bumiaji,1806915575,46265,96.17,96.17,High Impact
1869,Public WiFi Expansion,DISKOMINFO,Junrejo,9318680148,17033,95.92,95.92,High Impact
8756,Agricultural Innovation,Dinas Pertanian,Bumiaji,6418306353,29305,95.82,95.82,High Impact
7696,Healthcare Outreach,Dinas Kesehatan,Batu,3855391265,45813,95.76,95.76,High Impact
3950,Healthcare Outreach,Dinas Kesehatan,Batu,9147526091,23673,95.70,95.70,High Impact
6325,Public WiFi Expansion,DISKOMINFO,Bumiaji,1127110756,13807,95.67,95.67,High Impact
2559,Youth Entrepreneurship,Dinas Koperasi,Bumiaji,1605718202,49171,95.46,95.46,High Impact



📄 Executive Summary (JSON Output)
{
    "total_programs": 10000,
    "total_budget_allocated": 51788261574851.0,
    "average_effectiveness_score": 80.92,
    "average_roi_score": 80.92,
    "average_satisfaction_score": 4.09,
    "average_budget_utilization": 87.33,
    "high_impact_program_rate": 22.8,
    "top_program": "Environmental Sustainability",
    "top_department": "Dinas Lingkungan Hidup",
    "top_district": "Bumiaji",
    "government_program_status": "Strong Performance",
    "strategic_recommendations": [
        "Scale high-performing programs and redesign low-impact initiatives."
    ]
}
